# Adaptive k-MIP on Peptides (free-tier Colab)

Trains **fixed / linear / exponential** per-layer `k` schedules for GPS + k-MIP on **Peptides-func**.

**Free-tier tips**
- Runtime → Change runtime type → **T4 GPU**
- Mount Drive so disconnects don't wipe checkpoints
- Start with `MODE=quick` (20 epochs), then `standard` (100), then `full` (250) with resume
- LapPE/RWSE stay **off** (this is what made Colab OOM before)
- Naive top-k is fine for Peptides if pykeops fails

## 0) Runtime check

In [ ]:
import torch, psutil, os
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('vram_gb:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))
print('ram_gb:', round(psutil.virtual_memory().total/1e9, 2))
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → T4'

## 1) Mount Drive (recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/kmip_adaptive_k'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)

## 2) Get the code

Use **Option A** (upload zip from your Mac). Option B only works if you pushed *your* fork (upstream lacks adaptive-k).

In [ ]:
# Option A — upload zip produced by: bash colab/pack_for_colab.sh
from google.colab import files
import zipfile, shutil

REPO = '/content/kmip'
if os.path.exists(REPO):
    shutil.rmtree(REPO)
os.makedirs(REPO, exist_ok=True)

uploaded = files.upload()  # choose kmip-adaptive-k-colab.zip
assert len(uploaded) == 1, 'Upload exactly one zip'
zip_path = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(REPO)
# handle accidental nested folder
entries = [e for e in os.listdir(REPO) if not e.startswith('.')]
if len(entries) == 1 and os.path.isdir(os.path.join(REPO, entries[0])):
    nested = os.path.join(REPO, entries[0])
    if os.path.exists(os.path.join(nested, 'main.py')):
        for name in os.listdir(nested):
            shutil.move(os.path.join(nested, name), os.path.join(REPO, name))
        shutil.rmtree(nested)
assert os.path.exists(os.path.join(REPO, 'main.py')), 'main.py missing after unzip'
print('Repo ready at', REPO)

In [ ]:
# Option B — clone YOUR fork (skip if you used Option A)
# REPO = '/content/kmip'
# !rm -rf $REPO
# !git clone https://github.com/<you>/<your-fork>.git $REPO
# %cd $REPO

## 3) Point datasets + results at Drive

In [ ]:
%cd /content/kmip

# Persist dataset + results across disconnects
import os
DRIVE_ROOT = '/content/drive/MyDrive/kmip_adaptive_k'
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)

if os.path.islink('datasets') or os.path.exists('datasets'):
    if os.path.islink('datasets'):
        os.unlink('datasets')
    elif os.path.isdir('datasets') and not os.listdir('datasets'):
        os.rmdir('datasets')
if not os.path.exists('datasets'):
    os.symlink(f'{DRIVE_ROOT}/datasets', 'datasets')

print('datasets ->', os.path.realpath('datasets'))
print('cwd', os.getcwd())

## 4) Install environment

In [ ]:
!bash colab/setup_env.sh

## 5) Smoke test (1 epoch, tiny)

Confirms preprocess + one train step before long runs.

In [ ]:
!python main.py \
  --cfg configs/LRGB/Peptides-func/colab/GPS+kmip-fixed.yaml \
  wandb.use False \
  optim.max_epoch 1 \
  train.batch_size 32 \
  model.auto_satisfy_param_budget False \
  gt.layers 2 gnn.layers_mp 2 \
  gt.dim_hidden 64 gnn.dim_inner 64 \
  gt.sparse.val_dim 16 gt.sparse.kq_dim 8 \
  out_dir /content/drive/MyDrive/kmip_adaptive_k/results/smoke \
  name_tag smoke

## 6) Ablation runs (free-tier schedule)

| MODE | Epochs | When |
|------|--------|------|
| `quick` | 20 | first session / sanity |
| `standard` | 100 | main free-tier result |
| `full` | 250 | if quota allows; resumes from ckpt |

Re-run this cell after a disconnect — `train.auto_resume True` continues.

In [ ]:
# Start here on free tier:
import os
os.environ['MODE'] = 'quick'                      # quick | standard | full
os.environ['RUNS'] = 'fixed,linear,exponential'   # or e.g. 'linear' only
os.environ['OUT_ROOT'] = '/content/drive/MyDrive/kmip_adaptive_k/results'
os.environ['SEED'] = '0'

!bash colab/run_ablation.sh

## 7) Summarize best val/test AP from logs

In [ ]:
import re, pathlib, json
root = pathlib.Path('/content/drive/MyDrive/kmip_adaptive_k/results')
pat = re.compile(
    r'Best so far: epoch (\d+).*?val_ap: ([0-9.]+).*?test_ap: ([0-9.]+)'
)
rows = []
for log in sorted(root.rglob('train_*.log')):
    text = log.read_text(errors='ignore')
    matches = pat.findall(text)
    if not matches:
        continue
    epoch, vap, tap = matches[-1]
    rows.append({
        'run': log.stem.replace('train_', ''),
        'log': str(log),
        'best_epoch': int(epoch),
        'val_ap': float(vap),
        'test_ap': float(tap),
    })

import pandas as pd
df = pd.DataFrame(rows).sort_values('val_ap', ascending=False)
display(df)
out = root / 'summary.json'
out.write_text(json.dumps(rows, indent=2))
print('wrote', out)

## Free-tier game plan

1. Session 1: setup + smoke + `MODE=quick` (all three runs)
2. Session 2+: set `MODE=standard`, same `OUT_ROOT` (auto-resume)
3. Optional: `MODE=full` if you still have GPU time
4. Keep only one long run at a time if RAM gets tight (`RUNS=linear`)

Expected schedules (8 layers):
- **fixed:** `[15]*8`
- **linear:** `45 → 5`
- **exponential:** `45 * 0.7^ℓ`, min 5